# Human Cleanup Detector

Пайплайн как у детекции дефекации собак, но для **поз человека**:
определяет, **убрал ли человек за собакой**.

## Архитектуры для сравнения
- Recurrent: `LSTM`, `BiLSTM`, `GRU`, `BiGRU`
- SSM: `MambaLite` (pure PyTorch)
- **Transformers**: `TransformerEncoder` (CLS-token) и `TemporalTransformer` (PoseFormer-lite)

## Также
- Pretrained human pose: YOLO pose COCO-17 (`yolov8n/s-pose`, `yolo11n-pose`)
- Трекеры: IoU / DeepSORT
- Метрики, сохранение в `models/`, инференс на видео

> Точного публичного видеодатасета «уборка за собакой» почти нет.
> Proxies: Stipra, Charades throw/put/tidy, ShitSpotter (фото) + synthetic + свои ролики.


In [8]:
# @title Установка зависимостей
!pip install -q ultralytics opencv-python-headless tqdm scikit-learn matplotlib seaborn pandas deep_sort_realtime huggingface_hub requests

import os, sys, json, math, time, random, zipfile, shutil
from pathlib import Path
from collections import deque
from dataclasses import dataclass, asdict
from datetime import datetime

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score
)
import pandas as pd
from tqdm.auto import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

DEVICE: cpu
DEVICE: cpu


In [9]:
# @title Config
@dataclass
class Config:
    pose_models: tuple = ('yolov8n-pose.pt', 'yolov8s-pose.pt', 'yolo11n-pose.pt')
    num_keypoints: int = 17
    window_size: int = 48
    person_conf: float = 0.4
    batch_size: int = 32
    epochs: int = 12
    lr: float = 1e-3
    weight_decay: float = 1e-4
    seed: int = 42
    repo_root: str = ''
    data_root: str = ''
    models_root: str = ''
    videos_pos: str = ''
    videos_neg: str = ''

cfg = Config()
if Path('/content').exists():
    cfg.repo_root = '/content/dogsproject'
else:
    here = Path.cwd()
    cfg.repo_root = str(here.parent if here.name == 'cleanup_detection' else here)

cfg.data_root = str(Path(cfg.repo_root) / 'datasets')
cfg.models_root = str(Path(cfg.repo_root) / 'cleanup_detection' / 'models')
cfg.videos_pos = str(Path(cfg.data_root) / 'labeled_videos' / 'cleanup')
cfg.videos_neg = str(Path(cfg.data_root) / 'labeled_videos' / 'no_cleanup')
for p in [cfg.models_root, cfg.videos_pos, cfg.videos_neg]:
    Path(p).mkdir(parents=True, exist_ok=True)

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))

{
  "pose_models": [
    "yolov8n-pose.pt",
    "yolov8s-pose.pt",
    "yolo11n-pose.pt"
  ],
  "num_keypoints": 17,
  "window_size": 48,
  "person_conf": 0.4,
  "batch_size": 32,
  "epochs": 12,
  "lr": 0.001,
  "weight_decay": 0.0001,
  "seed": 42,
  "repo_root": "/content/dogsproject",
  "data_root": "/content/dogsproject/datasets",
  "models_root": "/content/dogsproject/cleanup_detection/models",
  "videos_pos": "/content/dogsproject/datasets/labeled_videos/cleanup",
  "videos_neg": "/content/dogsproject/datasets/labeled_videos/no_cleanup"
}
{
  "pose_models": [
    "yolov8n-pose.pt",
    "yolov8s-pose.pt",
    "yolo11n-pose.pt"
  ],
  "num_keypoints": 17,
  "window_size": 48,
  "person_conf": 0.4,
  "batch_size": 32,
  "epochs": 12,
  "lr": 0.001,
  "weight_decay": 0.0001,
  "seed": 42,
  "repo_root": "/content/dogsproject",
  "data_root": "/content/dogsproject/datasets",
  "models_root": "/content/dogsproject/cleanup_detection/models",
  "videos_pos": "/content/dogsproject/datase

## Датасеты
| Источник | Тип | Зачем |
|---|---|---|
| Stipra (~22GB, 1273 видео) | выброс мусора | лучший open proxy cleanup-движения |
| Charades | throw/put/tidy | proxy наклона/выброса |
| ShitSpotter | фото before/after | домен, не action |
| Свои CCTV | cleanup / no_cleanup | целевой датасет |


In [10]:
# @title Load / bootstrap data
import importlib.util
import shutil
import subprocess

# --- sync repo in Colab ---
REPO_URL = 'https://github.com/SemenSazhin/dog_system_training.git'
if Path('/content').exists():
    repo = Path('/content/dogsproject')
    if (repo / '.git').exists():
        # folder already cloned: update instead of clone
        subprocess.run(['git', '-C', str(repo), 'fetch', '--depth', '1', 'origin', 'master'], check=False)
        subprocess.run(['git', '-C', str(repo), 'reset', '--hard', 'origin/master'], check=False)
        print('repo synced:', repo)
    else:
        # leftover /content/dogsproject without .git -> clone fails with exit 128
        if repo.exists():
            print('removing non-git leftover:', repo)
            shutil.rmtree(repo)
        r = subprocess.run(
            ['git', 'clone', '--depth', '1', REPO_URL, str(repo)],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print('git clone failed:', (r.stderr or r.stdout or '')[-500:])
            raise RuntimeError(f'git clone failed with code {r.returncode}')
        print('repo cloned:', repo)
    cfg.repo_root = str(repo)
    cfg.data_root = str(repo / 'datasets')
    cfg.models_root = str(repo / 'cleanup_detection' / 'models')
    Path(cfg.models_root).mkdir(parents=True, exist_ok=True)

DATA = Path(cfg.data_root)
PROXY = DATA / 'cleanup_proxy' / 'synthetic_pose_sequences.npz'
MANIFEST = DATA / 'manifests' / 'DATASETS_CATALOG.json'
PROXY.parent.mkdir(parents=True, exist_ok=True)
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def generate_synthetic_pose_sequences_inline(n_pos=120, n_neg=120, window=48, n_kp=17, out_path=PROXY):
    # Local fallback: do not import package named datasets (conflicts with HF datasets)
    rng = np.random.default_rng(42)

    def base_pose(t, mode):
        kp = np.zeros((n_kp, 3), dtype=np.float32)
        kp[0] = [0.50, 0.18, 0.95]
        kp[5] = [0.45, 0.30, 0.95]
        kp[6] = [0.55, 0.30, 0.95]
        kp[11] = [0.46, 0.55, 0.95]
        kp[12] = [0.54, 0.55, 0.95]
        if mode == 'cleanup':
            squat = 0.12 * np.sin(np.pi * min(max((t - 0.2) / 0.5, 0.0), 1.0))
            reach = 0.20 * np.sin(np.pi * min(max((t - 0.25) / 0.45, 0.0), 1.0))
            kp[[5, 6, 11, 12], 1] += squat
            kp[0, 1] += squat * 0.8
            kp[9] = [0.42, 0.70 + reach, 0.9]
            kp[10] = [0.58, 0.70 + reach, 0.9]
            kp[7] = [0.43, 0.50 + reach * 0.5, 0.9]
            kp[8] = [0.57, 0.50 + reach * 0.5, 0.9]
            kp[13] = [0.45, 0.75 + squat, 0.9]
            kp[14] = [0.55, 0.75 + squat, 0.9]
            kp[15] = [0.44, 0.92, 0.9]
            kp[16] = [0.56, 0.92, 0.9]
        else:
            sway = 0.03 * np.sin(2 * np.pi * t * 2)
            kp[:, 0] += sway
            kp[9] = [0.40, 0.48, 0.9]
            kp[10] = [0.60, 0.48, 0.9]
            kp[7] = [0.42, 0.40, 0.9]
            kp[8] = [0.58, 0.40, 0.9]
            kp[13] = [0.46, 0.72 + 0.02 * np.sin(2 * np.pi * t * 3), 0.9]
            kp[14] = [0.54, 0.72 + 0.02 * np.cos(2 * np.pi * t * 3), 0.9]
            kp[15] = [0.45, 0.92, 0.9]
            kp[16] = [0.56, 0.92, 0.9]
        noise = rng.normal(0, 0.008, size=kp.shape).astype(np.float32)
        noise[:, 2] = 0
        kp = np.clip(kp + noise, 0.0, 1.0)
        kp[:, 2] = np.clip(kp[:, 2], 0.5, 1.0)
        return kp

    def make_seq(label):
        mode = 'cleanup' if label == 1 else 'other'
        frames = [base_pose(i / max(window - 1, 1), mode).reshape(-1) for i in range(window)]
        return np.stack(frames, axis=0)

    X, y = [], []
    for _ in range(n_pos):
        X.append(make_seq(1)); y.append(1)
    for _ in range(n_neg):
        X.append(make_seq(0)); y.append(0)
    X_np = np.stack(X).astype(np.float32)
    y_np = np.asarray(y, dtype=np.int64)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(out_path, X=X_np, y=y_np)
    print('generated synthetic:', X_np.shape, '->', out_path)
    return X_np, y_np


# Optional helper via absolute path (avoid `import datasets`)
helper_path = Path(cfg.repo_root) / 'datasets' / 'download_datasets.py'
if helper_path.exists():
    try:
        spec = importlib.util.spec_from_file_location('cleanup_download_datasets', helper_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        if hasattr(mod, 'write_dataset_catalog'):
            mod.write_dataset_catalog()
        print('helper loaded from', helper_path)
    except Exception as e:
        print('helper load skipped:', e)

if not PROXY.exists():
    generate_synthetic_pose_sequences_inline()

if MANIFEST.exists():
    cat = json.loads(MANIFEST.read_text(encoding='utf-8'))
    print('pretrained pose:', cat.get('pretrained_human_pose'))
    print('datasets:', [d['id'] for d in cat.get('datasets', [])])
else:
    print('manifest missing (ok for bootstrap):', MANIFEST)

assert PROXY.exists(), f'Missing synthetic data: {PROXY}'
pack = np.load(PROXY)
X_syn, y_syn = pack['X'], pack['y']
print('synthetic', X_syn.shape, 'pos', int((y_syn == 1).sum()), 'neg', int((y_syn == 0).sum()))


removing non-git leftover: /content/dogsproject
removing non-git leftover: /content/dogsproject
repo cloned: /content/dogsproject
[ok] catalog -> /content/dogsproject/datasets/manifests/DATASETS_CATALOG.json
helper loaded from /content/dogsproject/datasets/download_datasets.py
pretrained pose: [{'name': 'YOLOv8n-pose / YOLOv8s-pose / YOLOv8m-pose', 'source': 'Ultralytics', 'weights': ['yolov8n-pose.pt', 'yolov8s-pose.pt', 'yolov8m-pose.pt'], 'keypoints': 17, 'train_data': 'COCO Keypoints'}, {'name': 'YOLOv11n-pose', 'source': 'Ultralytics', 'weights': ['yolo11n-pose.pt'], 'keypoints': 17, 'train_data': 'COCO Keypoints'}, {'name': 'MediaPipe Pose Landmarker', 'source': 'Google MediaPipe', 'keypoints': 33, 'note': 'Good CPU realtime alternative; different skeleton than COCO-17'}]
datasets: ['stipra', 'shitspotter', 'charades', 'kinetics_related', 'stair_actions']
synthetic (240, 48, 51) pos 120 neg 120
repo cloned: /content/dogsproject
[ok] catalog -> /content/dogsproject/datasets/manife

## Features + модели (включая Transformer)
COCO-17 → углы + относительные координаты + Δ.
Особо важны для cleanup: сгиб коленей, наклон корпуса, высота запястий.


In [11]:
# @title Features + architectures (LSTM/GRU/Mamba/Transformer)
L_SHO, R_SHO, L_ELB, R_ELB, L_WRI, R_WRI = 5, 6, 7, 8, 9, 10
L_HIP, R_HIP, L_KNE, R_KNE, L_ANK, R_ANK = 11, 12, 13, 14, 15, 16

KEY_ANGLES = [
    (L_SHO, L_ELB, L_WRI), (R_SHO, R_ELB, R_WRI),
    (L_HIP, L_KNE, L_ANK), (R_HIP, R_KNE, R_ANK),
    (L_SHO, L_HIP, L_KNE), (R_SHO, R_HIP, R_KNE),
    (L_ELB, L_SHO, L_HIP), (R_ELB, R_SHO, R_HIP),
]


def _angle(a, b, c):
    ba, bc = a - b, c - b
    n1, n2 = np.linalg.norm(ba) + 1e-8, np.linalg.norm(bc) + 1e-8
    cos = np.clip(np.dot(ba, bc) / (n1 * n2), -1.0, 1.0)
    return float(np.arccos(cos) / np.pi)


def pose_frame_features(kpts_xyv: np.ndarray) -> np.ndarray:
    xy = kpts_xyv[:, :2].astype(np.float32)
    conf = kpts_xyv[:, 2].astype(np.float32)
    mid_hip = (xy[L_HIP] + xy[R_HIP]) * 0.5
    mid_sho = (xy[L_SHO] + xy[R_SHO]) * 0.5
    scale = np.linalg.norm(mid_sho - mid_hip) + 1e-6
    rel = (xy - mid_hip) / scale
    feats = []
    feats.extend(rel.reshape(-1).tolist())
    feats.extend(conf.tolist())
    for i, j, k in KEY_ANGLES:
        feats.append(_angle(xy[i], xy[j], xy[k]))
    wrist_h = ((xy[L_WRI, 1] + xy[R_WRI, 1]) * 0.5 - mid_hip[1]) / scale
    torso_tilt = float(np.arctan2(mid_sho[0] - mid_hip[0], mid_hip[1] - mid_sho[1] + 1e-6) / np.pi)
    knee_bend = 0.5 * (_angle(xy[L_HIP], xy[L_KNE], xy[L_ANK]) + _angle(xy[R_HIP], xy[R_KNE], xy[R_ANK]))
    feats.extend([wrist_h, torso_tilt, knee_bend])
    return np.asarray(feats, dtype=np.float32)


def sequence_features(kpts_seq: np.ndarray) -> np.ndarray:
    Fmat = np.stack([pose_frame_features(k) for k in kpts_seq], axis=0)
    dlt = np.zeros_like(Fmat)
    dlt[1:] = Fmat[1:] - Fmat[:-1]
    return np.concatenate([Fmat, dlt], axis=1)


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=False, num_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        od = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(od, od // 2), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(od // 2, num_classes))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1])


class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, bidirectional=False, num_classes=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0, bidirectional=bidirectional)
        od = hidden_size * (2 if bidirectional else 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(od, od // 2), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(od // 2, num_classes))

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out[:, -1])


class _MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        d_inner = d_model * expand
        self.in_proj = nn.Linear(d_model, d_inner * 2)
        self.conv = nn.Conv1d(d_inner, d_inner, 3, padding=1, groups=d_inner)
        self.x_proj = nn.Linear(d_inner, d_state * 2 + d_inner)
        self.dt_proj = nn.Linear(d_inner, d_inner)
        self.out_proj = nn.Linear(d_inner, d_model)
        self.A_log = nn.Parameter(torch.log(torch.arange(1, d_state + 1, dtype=torch.float32)))
        self.D = nn.Parameter(torch.ones(d_inner))
        self.d_state, self.d_inner = d_state, d_inner

    def forward(self, x):
        xz = self.in_proj(x)
        x_b, z = xz.chunk(2, dim=-1)
        x_b = F.silu(self.conv(x_b.transpose(1, 2)).transpose(1, 2))
        dbl = self.x_proj(x_b)
        delta, B, C = torch.split(dbl, [self.d_inner, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))
        A = -torch.exp(self.A_log)
        ys = []
        h = torch.zeros(x.size(0), self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        for t in range(x.size(1)):
            dt = delta[:, t].unsqueeze(-1)
            a_bar = torch.exp(dt * A)
            b_bar = dt * B[:, t].unsqueeze(1)
            h = a_bar * h + b_bar * x_b[:, t].unsqueeze(-1)
            yt = torch.einsum('bdn,bn->bd', h, C[:, t]) + self.D * x_b[:, t]
            ys.append(yt)
        y = torch.stack(ys, 1) * F.silu(z)
        return self.out_proj(y)


class MambaLiteClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3, num_classes=2, d_state=16):
        super().__init__()
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.blocks = nn.ModuleList([_MambaBlock(hidden_size, d_state=d_state) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, hidden_size // 2), nn.GELU(),
                                  nn.Dropout(dropout), nn.Linear(hidden_size // 2, num_classes))

    def forward(self, x):
        h = self.input_proj(x)
        for b in self.blocks:
            h = h + b(self.norm(h))
        return self.head(h.mean(1))


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerEncoderClassifier(nn.Module):
    # Transformer encoder + CLS token
    def __init__(self, input_size, hidden_size=128, num_layers=2, nhead=4, dropout=0.3, num_classes=2, max_len=512):
        super().__init__()
        if hidden_size % nhead:
            hidden_size = nhead * max(1, hidden_size // nhead)
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.cls = nn.Parameter(torch.zeros(1, 1, hidden_size))
        nn.init.normal_(self.cls, std=0.02)
        self.pos = PositionalEncoding(hidden_size, max_len + 1, dropout)
        layer = nn.TransformerEncoderLayer(
            hidden_size, nhead, hidden_size * 4, dropout, 'gelu',
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden_size, hidden_size // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_size // 2, num_classes),
        )

    def forward(self, x):
        h = self.input_proj(x)
        h = torch.cat([self.cls.expand(x.size(0), -1, -1), h], 1)
        h = self.encoder(self.pos(h))
        return self.head(self.norm(h[:, 0]))


class TemporalTransformerClassifier(nn.Module):
    # PoseFormer-lite: temporal Transformer + mean/max pooling
    def __init__(self, input_size, hidden_size=128, num_layers=3, nhead=4, dropout=0.3, num_classes=2, max_len=512):
        super().__init__()
        if hidden_size % nhead:
            hidden_size = nhead * max(1, hidden_size // nhead)
        self.input_proj = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.LayerNorm(hidden_size), nn.GELU(), nn.Dropout(dropout)
        )
        self.pos = PositionalEncoding(hidden_size, max_len, dropout)
        layer = nn.TransformerEncoderLayer(
            hidden_size, nhead, hidden_size * 4, dropout, 'gelu',
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.norm = nn.LayerNorm(hidden_size)
        self.head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(hidden_size * 2, hidden_size), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        h = self.norm(self.encoder(self.pos(self.input_proj(x))))
        return self.head(torch.cat([h.mean(1), h.max(1).values], -1))


def build_model(name, input_size, **kw):
    name = name.lower()
    common = {k: kw[k] for k in ('hidden_size', 'num_layers', 'dropout', 'num_classes') if k in kw}
    if name == 'lstm':
        return LSTMClassifier(input_size, bidirectional=False, **common)
    if name == 'bilstm':
        return LSTMClassifier(input_size, bidirectional=True, **common)
    if name == 'gru':
        return GRUClassifier(input_size, bidirectional=False, **common)
    if name == 'bigru':
        return GRUClassifier(input_size, bidirectional=True, **common)
    if name == 'mamba_lite':
        return MambaLiteClassifier(input_size, **common)
    if name == 'transformer':
        tkw = dict(common)
        if 'nhead' in kw:
            tkw['nhead'] = kw['nhead']
        return TransformerEncoderClassifier(input_size, **tkw)
    if name == 'temporal_transformer':
        tkw = dict(common)
        if 'nhead' in kw:
            tkw['nhead'] = kw['nhead']
        return TemporalTransformerClassifier(input_size, **tkw)
    raise KeyError(name)


print('Architectures ready: lstm, bilstm, gru, bigru, mamba_lite, transformer, temporal_transformer')

Architectures ready: lstm, bilstm, gru, bigru, mamba_lite, transformer, temporal_transformer


## Обучение и сравнение архитектур
Включая **transformer** и **temporal_transformer**.


In [12]:
# @title Prepare splits + train/eval
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]


def prepare_xy(X, y):
    # synthetic stores flattened COCO-17 xyz => reshape then featureize
    T = cfg.window_size
    if X.shape[-1] == 17 * 3:
        k = X.reshape(len(X), T, 17, 3)
        Xs = np.stack([sequence_features(ki) for ki in k]).astype(np.float32)
    else:
        Xs = X.astype(np.float32)
    return Xs, y.astype(np.int64)


X_all, y_all = prepare_xy(X_syn, y_syn)
INPUT_SIZE = X_all.shape[-1]
print('X_all', X_all.shape)

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=cfg.seed, stratify=y_all
)
X_tr, X_va, y_tr, y_va = train_test_split(
    X_tmp, y_tmp, test_size=0.2, random_state=cfg.seed, stratify=y_tmp
)

train_loader = DataLoader(SeqDataset(X_tr, y_tr), batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(SeqDataset(X_va, y_va), batch_size=cfg.batch_size)
test_loader = DataLoader(SeqDataset(X_test, y_test), batch_size=cfg.batch_size)


def evaluate(model, loader):
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(DEVICE))
            p = torch.softmax(logits, 1)[:, 1].cpu().numpy()
            pr = logits.argmax(1).cpu().numpy()
            preds.extend(pr.tolist())
            trues.extend(yb.numpy().tolist())
            probs.extend(p.tolist())
    acc = accuracy_score(trues, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(trues, preds, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(trues, probs)
    except Exception:
        auc = float('nan')
    return {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'roc_auc': auc,
        'report': classification_report(trues, preds, target_names=['no_cleanup', 'cleanup'], zero_division=0),
        'cm': confusion_matrix(trues, preds).tolist(),
        'samples': [
            {'i': i, 'true': int(trues[i]), 'pred': int(preds[i]), 'prob_cleanup': round(float(probs[i]), 4)}
            for i in range(min(3, len(trues)))
        ],
    }


def train_one(tag, arch, params, epochs=None):
    epochs = epochs or cfg.epochs
    model_kwargs = {k: v for k, v in params.items() if k not in ('lr', 'weight_decay')}
    model = build_model(arch, INPUT_SIZE, **model_kwargs).to(DEVICE)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=params.get('lr', cfg.lr),
        weight_decay=params.get('weight_decay', cfg.weight_decay),
    )
    crit = nn.CrossEntropyLoss()
    best_f1, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            losses.append(loss.item())
        val = evaluate(model, val_loader)
        print(f"[{tag}] ep {ep+1}/{epochs} loss={np.mean(losses):.4f} val_f1={val['f1']:.4f}")
        if val['f1'] >= best_f1:
            best_f1 = val['f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    test = evaluate(model, test_loader)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    exp_dir = Path(cfg.models_root) / tag
    if exp_dir.exists():
        shutil.rmtree(exp_dir)
    exp_dir.mkdir(parents=True, exist_ok=True)
    torch.save({
        'model_state_dict': model.state_dict(),
        'arch': arch,
        'params': params,
        'input_size': INPUT_SIZE,
        'metrics': {k: test[k] for k in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'cm')},
    }, exp_dir / 'model.pth')
    md_lines = [
        f'# {tag}', '',
        f'- arch: {arch}',
        f"- accuracy: {test['accuracy']:.4f}",
        f"- precision: {test['precision']:.4f}",
        f"- recall: {test['recall']:.4f}",
        f"- f1: {test['f1']:.4f}",
        f"- roc_auc: {test['roc_auc']:.4f}",
        f'- params: {n_params}', '',
        '## Hyperparams',
    ] + [f'- {k}: {v}' for k, v in params.items()] + [
        '', '## Confusion matrix', str(test['cm']), '',
        '## Report', '```', test['report'], '```', '',
        '## 3 samples',
    ] + [f'- {s}' for s in test['samples']]
    (exp_dir / 'metrics.md').write_text('\n'.join(md_lines), encoding='utf-8')
    return {
        'tag': tag, 'arch': arch, 'n_params': n_params, 'dir': str(exp_dir),
        'accuracy': test['accuracy'], 'precision': test['precision'],
        'recall': test['recall'], 'f1': test['f1'], 'roc_auc': test['roc_auc'],
        'lr': params.get('lr', cfg.lr),
    }


EXPERIMENTS = [
    ('lstm_base', 'lstm', dict(hidden_size=128, num_layers=2, dropout=0.3, num_classes=2, lr=1e-3)),
    ('bilstm_base', 'bilstm', dict(hidden_size=128, num_layers=2, dropout=0.3, num_classes=2, lr=1e-3)),
    ('gru_base', 'gru', dict(hidden_size=128, num_layers=2, dropout=0.3, num_classes=2, lr=1e-3)),
    ('bigru_base', 'bigru', dict(hidden_size=128, num_layers=2, dropout=0.3, num_classes=2, lr=8e-4)),
    ('mamba_lite_base', 'mamba_lite', dict(hidden_size=128, num_layers=2, dropout=0.3, num_classes=2, lr=1e-3)),
    # Transformers
    ('transformer_cls', 'transformer', dict(hidden_size=128, num_layers=2, nhead=4, dropout=0.3, num_classes=2, lr=8e-4)),
    ('transformer_wide', 'transformer', dict(hidden_size=192, num_layers=3, nhead=6, dropout=0.35, num_classes=2, lr=5e-4)),
    ('temporal_transformer', 'temporal_transformer', dict(hidden_size=128, num_layers=3, nhead=4, dropout=0.3, num_classes=2, lr=7e-4)),
]

results = []
for tag, arch, params in EXPERIMENTS:
    print('\n' + '=' * 70)
    print('RUN', tag)
    results.append(train_one(tag, arch, params))

df = pd.DataFrame(results).sort_values('f1', ascending=False)
print('\n=== ARCHITECTURE COMPARISON (incl. Transformers) ===')
print(df.to_string(index=False))
best = max(results, key=lambda r: r['f1'])
print('BEST:', best['tag'], 'F1=', round(best['f1'], 4))
df.to_csv(Path(cfg.models_root) / 'architecture_comparison.csv', index=False)
(Path(cfg.models_root) / 'summary.md').write_text(
    '# Architecture comparison\n\n' + df.to_markdown(index=False) +
    f"\n\nBest: **{best['tag']}** ({best['arch']}), F1={best['f1']:.4f}\n",
    encoding='utf-8',
)

X_all (240, 48, 124)

RUN lstm_base
[lstm_base] ep 1/12 loss=0.6874 val_f1=1.0000
[lstm_base] ep 2/12 loss=0.6420 val_f1=1.0000
[lstm_base] ep 3/12 loss=0.3872 val_f1=1.0000
[lstm_base] ep 4/12 loss=0.0986 val_f1=1.0000
[lstm_base] ep 5/12 loss=0.0192 val_f1=1.0000
[lstm_base] ep 6/12 loss=0.0044 val_f1=1.0000
[lstm_base] ep 7/12 loss=0.0011 val_f1=1.0000
[lstm_base] ep 8/12 loss=0.0005 val_f1=1.0000
[lstm_base] ep 9/12 loss=0.0003 val_f1=1.0000
[lstm_base] ep 10/12 loss=0.0002 val_f1=1.0000
[lstm_base] ep 11/12 loss=0.0002 val_f1=1.0000
[lstm_base] ep 12/12 loss=0.0001 val_f1=1.0000

RUN bilstm_base
[bilstm_base] ep 1/12 loss=0.6885 val_f1=1.0000
[bilstm_base] ep 2/12 loss=0.6407 val_f1=1.0000
[bilstm_base] ep 3/12 loss=0.3457 val_f1=1.0000
[bilstm_base] ep 4/12 loss=0.0871 val_f1=1.0000
[bilstm_base] ep 5/12 loss=0.0024 val_f1=1.0000
[bilstm_base] ep 6/12 loss=0.0034 val_f1=1.0000
[bilstm_base] ep 7/12 loss=0.0001 val_f1=1.0000
[bilstm_base] ep 8/12 loss=0.0000 val_f1=1.0000
[bilstm_

/tmp/ipykernel_8453/1845424149.py:153: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers)


[transformer_cls] ep 1/12 loss=0.7159 val_f1=1.0000
[transformer_cls] ep 2/12 loss=0.6891 val_f1=0.6552
[transformer_cls] ep 3/12 loss=0.6210 val_f1=1.0000
[transformer_cls] ep 4/12 loss=0.4541 val_f1=1.0000
[transformer_cls] ep 5/12 loss=0.2052 val_f1=1.0000
[transformer_cls] ep 6/12 loss=0.0662 val_f1=1.0000
[transformer_cls] ep 7/12 loss=0.0337 val_f1=1.0000
[transformer_cls] ep 8/12 loss=0.0131 val_f1=1.0000
[transformer_cls] ep 9/12 loss=0.0073 val_f1=1.0000
[transformer_cls] ep 10/12 loss=0.0045 val_f1=1.0000
[transformer_cls] ep 11/12 loss=0.0038 val_f1=1.0000
[transformer_cls] ep 12/12 loss=0.0024 val_f1=1.0000

RUN transformer_wide


/tmp/ipykernel_8453/1845424149.py:153: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers)


[transformer_wide] ep 1/12 loss=0.6974 val_f1=0.0000
[transformer_wide] ep 2/12 loss=0.6350 val_f1=1.0000
[transformer_wide] ep 3/12 loss=0.4670 val_f1=1.0000
[transformer_wide] ep 4/12 loss=0.1791 val_f1=1.0000
[transformer_wide] ep 5/12 loss=0.0427 val_f1=1.0000
[transformer_wide] ep 6/12 loss=0.0144 val_f1=1.0000
[transformer_wide] ep 7/12 loss=0.0077 val_f1=1.0000
[transformer_wide] ep 8/12 loss=0.0038 val_f1=1.0000
[transformer_wide] ep 9/12 loss=0.0028 val_f1=1.0000
[transformer_wide] ep 10/12 loss=0.0019 val_f1=1.0000
[transformer_wide] ep 11/12 loss=0.0016 val_f1=1.0000
[transformer_wide] ep 12/12 loss=0.0011 val_f1=1.0000

RUN temporal_transformer


/tmp/ipykernel_8453/1845424149.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers)


[temporal_transformer] ep 1/12 loss=0.6866 val_f1=1.0000
[temporal_transformer] ep 2/12 loss=0.4670 val_f1=1.0000
[temporal_transformer] ep 3/12 loss=0.0682 val_f1=1.0000
[temporal_transformer] ep 4/12 loss=0.0047 val_f1=1.0000
[temporal_transformer] ep 5/12 loss=0.0017 val_f1=1.0000
[temporal_transformer] ep 6/12 loss=0.0007 val_f1=1.0000
[temporal_transformer] ep 7/12 loss=0.0005 val_f1=1.0000
[temporal_transformer] ep 8/12 loss=0.0004 val_f1=1.0000
[temporal_transformer] ep 9/12 loss=0.0003 val_f1=1.0000
[temporal_transformer] ep 10/12 loss=0.0002 val_f1=1.0000
[temporal_transformer] ep 11/12 loss=0.0002 val_f1=1.0000
[temporal_transformer] ep 12/12 loss=0.0002 val_f1=1.0000

=== ARCHITECTURE COMPARISON (incl. Transformers) ===
                 tag                 arch  n_params                                                                dir  accuracy  precision  recall  f1  roc_auc     lr
           lstm_base                 lstm    270530            /content/dogsproject/cleanup

2026

## Pose backends + трекеры
Комбинации YOLO-pose весов и трекеров (IoU / DeepSORT) на видео.


In [13]:
# @title Pose/tracker combo smoke-test
from ultralytics import YOLO

try:
    from deep_sort_realtime.deepsort_tracker import DeepSort
    HAS_DEEPSORT = True
except Exception:
    HAS_DEEPSORT = False
    print('DeepSORT unavailable')


class IoUTracker:
    def __init__(self, iou_thr=0.3, max_age=20):
        self.iou_thr, self.max_age = iou_thr, max_age
        self.tracks, self._next = {}, 1

    @staticmethod
    def iou(a, b):
        x1, y1 = max(a[0], b[0]), max(a[1], b[1])
        x2, y2 = min(a[2], b[2]), min(a[3], b[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-6
        return inter / ua

    def update(self, boxes):
        assigned, alive = set(), {}
        for tid, tr in self.tracks.items():
            best, bi = 0, None
            for i, b in enumerate(boxes):
                if i in assigned:
                    continue
                v = self.iou(tr['box'], b)
                if v > best:
                    best, bi = v, i
            if bi is not None and best >= self.iou_thr:
                alive[tid] = {'box': boxes[bi], 'age': 0}
                assigned.add(bi)
            elif tr['age'] + 1 <= self.max_age:
                alive[tid] = {'box': tr['box'], 'age': tr['age'] + 1}
        for i, b in enumerate(boxes):
            if i not in assigned:
                alive[self._next] = {'box': b, 'age': 0}
                self._next += 1
        self.tracks = alive
        return [(tid, t['box']) for tid, t in self.tracks.items() if t['age'] == 0]


def extract_person_pose_sequence(video_path, pose_weights='yolov8n-pose.pt', tracker='iou', max_frames=240):
    pose = YOLO(pose_weights)
    track = IoUTracker()
    ds = DeepSort(max_age=30) if (tracker == 'deepsort' and HAS_DEEPSORT) else None
    cap = cv2.VideoCapture(str(video_path))
    seqs = {}
    frames = 0
    while frames < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        frames += 1
        res = pose(frame, verbose=False)[0]
        boxes, kps = [], []
        if res.boxes is not None and len(res.boxes):
            for i in range(len(res.boxes)):
                if float(res.boxes.conf[i]) < cfg.person_conf:
                    continue
                boxes.append(res.boxes.xyxy[i].cpu().numpy().tolist())
                kps.append(res.keypoints.data[i].cpu().numpy() if res.keypoints is not None else np.zeros((17, 3), np.float32))
        if ds is not None:
            dets = [([b[0], b[1], b[2] - b[0], b[3] - b[1]], 0.9, 'person') for b in boxes]
            for tr in ds.update_tracks(dets, frame=frame):
                if not tr.is_confirmed() or not boxes:
                    continue
                l, t, r, btm = tr.to_ltrb()
                i = int(np.argmin([abs((b[0] + b[2]) / 2 - (l + r) / 2) for b in boxes]))
                seqs.setdefault(tr.track_id, []).append(kps[i])
        else:
            for tid, box in track.update(boxes):
                if not boxes:
                    continue
                i = int(np.argmin([abs(sum(np.array(box) - np.array(bb))) for bb in boxes]))
                seqs.setdefault(tid, []).append(kps[i])
    cap.release()
    if not seqs:
        return None
    tid = max(seqs, key=lambda k: len(seqs[k]))
    arr = np.stack(seqs[tid], 0)
    if len(arr) < cfg.window_size:
        pad = np.repeat(arr[-1:], cfg.window_size - len(arr), 0)
        arr = np.concatenate([arr, pad], 0)
    else:
        arr = arr[-cfg.window_size:]
    return sequence_features(arr)


video_candidates = list(Path(cfg.videos_pos).glob('*.mp4')) + list(Path(cfg.videos_neg).glob('*.mp4'))
combo_rows = []
if video_candidates:
    v = video_candidates[0]
    for pose_w in cfg.pose_models:
        for tr in (['iou', 'deepsort'] if HAS_DEEPSORT else ['iou']):
            t0 = time.time()
            try:
                feat = extract_person_pose_sequence(v, pose_w, tr, max_frames=90)
                ok, shape = feat is not None, (None if feat is None else list(feat.shape))
            except Exception as e:
                ok, shape = False, str(e)
            row = {'pose': pose_w, 'tracker': tr, 'ok': ok, 'feat_shape': shape, 'sec': round(time.time() - t0, 2)}
            combo_rows.append(row)
            print(row)
    pd.DataFrame(combo_rows).to_csv(Path(cfg.models_root) / 'pose_tracker_combos.csv', index=False)
else:
    print('Нет mp4 в labeled_videos. Добавьте ролики в:')
    print(cfg.videos_pos)
    print(cfg.videos_neg)
    print('Pretrained pose: yolov8n-pose.pt / yolov8s-pose.pt / yolo11n-pose.pt')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Нет mp4 в labeled_videos. Добавьте ролики в:
/content/dogsproject/datasets/labeled_videos/cleanup
/content/dogsproject/datasets/labeled_videos/no_cleanup
Pretrained pose: yolov8n-pose.pt / yolov8s-pose.pt / yolo11n-pose.pt


## Инференс всех архитектур на видео


In [17]:
# @title Video inference (all architectures)
def resolve_arch(name: str) -> str:
    for a in ['temporal_transformer', 'transformer', 'mamba_lite', 'bilstm', 'bigru', 'lstm', 'gru']:
        if name.startswith(a) or a in name:
            return a
    return name


def load_sequence_model(row):
    ckpt_path = Path(row['dir']) / 'model.pth'
    # PyTorch 2.6+ defaults weights_only=True; our ckpt stores numpy scalars in metrics/params
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    arch = resolve_arch(ckpt.get('arch', row['arch']))
    params = {k: v for k, v in ckpt.get('params', {}).items() if k not in ('lr', 'weight_decay')}
    model = build_model(arch, ckpt['input_size'], **params).to(DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, ckpt


def load_best_sequence_model():
    return load_sequence_model(best)


def extract_pose_stream(video_path, pose_weights='yolov8n-pose.pt'):
    """Run YOLO-pose once; yield frames for all models."""
    pose = YOLO(pose_weights)
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f'Cannot open video: {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    w, h = int(cap.get(3)), int(cap.get(4))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        res = pose(frame, verbose=False)[0]
        boxes, kps = [], []
        if res.boxes is not None:
            for i in range(len(res.boxes)):
                if float(res.boxes.conf[i]) < cfg.person_conf:
                    continue
                boxes.append(res.boxes.xyxy[i].cpu().numpy().tolist())
                if res.keypoints is not None:
                    kps.append(res.keypoints.data[i].cpu().numpy())
                else:
                    kps.append(np.zeros((17, 3), np.float32))
        frames.append((frame, boxes, kps))
    cap.release()
    return frames, fps, w, h


def run_video_cleanup_detector(
    video_path,
    output_path,
    model=None,
    row=None,
    pose_weights='yolov8n-pose.pt',
    thr=0.5,
    pose_cache=None,
    tag='',
):
    if model is None:
        model, _ = load_sequence_model(row or best)
    if pose_cache is None:
        frames, fps, w, h = extract_pose_stream(video_path, pose_weights)
    else:
        frames, fps, w, h = pose_cache

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    track = IoUTracker()
    buffers = {}
    probs_hist = []
    n_cleanup = 0
    n_scored = 0

    for frame, boxes, kps in frames:
        vis = frame.copy()
        label, p = 'no_cleanup', 0.0
        for tid, box in track.update(boxes):
            if not boxes:
                continue
            i = int(np.argmin([abs(sum(np.array(box) - np.array(bb))) for bb in boxes]))
            buffers.setdefault(tid, deque(maxlen=cfg.window_size)).append(kps[i])
            if len(buffers[tid]) == cfg.window_size:
                feat = sequence_features(np.stack(buffers[tid], 0))
                with torch.no_grad():
                    logits = model(torch.tensor(feat, dtype=torch.float32, device=DEVICE).unsqueeze(0))
                    p = float(torch.softmax(logits, 1)[0, 1].cpu())
                label = 'cleanup' if p >= thr else 'no_cleanup'
                probs_hist.append(p)
                n_scored += 1
                n_cleanup += int(label == 'cleanup')
            x1, y1, x2, y2 = map(int, box)
            color = (0, 200, 0) if label == 'cleanup' else (0, 165, 255)
            title = f'{tag} id:{tid} {label} {p:.2f}'.strip()
            cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
            cv2.putText(vis, title, (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
        writer.write(vis)
    writer.release()

    mean_p = float(np.mean(probs_hist)) if probs_hist else 0.0
    cleanup_rate = float(n_cleanup / n_scored) if n_scored else 0.0
    print(f"saved {output_path} | mean_p={mean_p:.3f} cleanup_rate={cleanup_rate:.3f} scored={n_scored}")
    return {
        'output_path': str(output_path),
        'mean_p': mean_p,
        'cleanup_rate': cleanup_rate,
        'n_scored_windows': n_scored,
        'n_cleanup_windows': n_cleanup,
    }


def run_video_all_architectures(
    video_path='/content/test.mp4',
    out_dir=None,
    pose_weights='yolov8n-pose.pt',
    thr=0.5,
    model_rows=None,
):
    model_rows = model_rows or results
    out_dir = Path(out_dir or (Path(cfg.models_root) / 'video_runs'))
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f'Extracting pose once from {video_path} ...')
    frames, fps, w, h = extract_pose_stream(video_path, pose_weights)
    pose_cache = (frames, fps, w, h)
    print(f'frames={len(frames)} fps={fps:.2f} size={w}x{h}; running {len(model_rows)} models')

    summary = []
    for row in model_rows:
        tag = row['tag']
        print(f'\n=== {tag} ({row["arch"]}) ===')
        model, _ = load_sequence_model(row)
        out_path = out_dir / f'{tag}.mp4'
        stats = run_video_cleanup_detector(
            video_path,
            out_path,
            model=model,
            thr=thr,
            pose_cache=pose_cache,
            tag=tag,
        )
        summary.append({
            'tag': tag,
            'arch': row['arch'],
            'train_f1': row.get('f1'),
            'train_accuracy': row.get('accuracy'),
            **stats,
            'is_best': tag == best['tag'],
        })
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    sdf = pd.DataFrame(summary).sort_values('mean_p', ascending=False)
    csv_path = out_dir / 'video_inference_summary.csv'
    sdf.to_csv(csv_path, index=False)
    print('\n=== VIDEO INFERENCE ALL ARCHITECTURES ===')
    print(sdf.to_string(index=False))
    print('summary:', csv_path)
    print('best-by-train still:', best['tag'])
    return sdf


print('Models for video test:', [r['tag'] for r in results])
print('Best-by-train F1:', best['tag'], best['arch'])
# Uncomment / set path:
video_summary = run_video_all_architectures('/content/test.mp4')
# # single model still available:
# run_video_cleanup_detector('/content/test.mp4', '/content/cleanup_out_best.mp4', row=best)


Models for video test: ['lstm_base', 'bilstm_base', 'gru_base', 'bigru_base', 'mamba_lite_base', 'transformer_cls', 'transformer_wide', 'temporal_transformer']
Best-by-train F1: lstm_base lstm
Extracting pose once from /content/test.mp4 ...
frames=1197 fps=59.97 size=720x1280; running 8 models

=== lstm_base (lstm) ===
saved /content/dogsproject/cleanup_detection/models/video_runs/lstm_base.mp4 | mean_p=0.271 cleanup_rate=0.273 scored=465

=== bilstm_base (bilstm) ===
saved /content/dogsproject/cleanup_detection/models/video_runs/bilstm_base.mp4 | mean_p=0.260 cleanup_rate=0.260 scored=465

=== gru_base (gru) ===
saved /content/dogsproject/cleanup_detection/models/video_runs/gru_base.mp4 | mean_p=0.146 cleanup_rate=0.146 scored=465

=== bigru_base (bigru) ===
saved /content/dogsproject/cleanup_detection/models/video_runs/bigru_base.mp4 | mean_p=0.164 cleanup_rate=0.163 scored=465

=== mamba_lite_base (mamba_lite) ===
saved /content/dogsproject/cleanup_detection/models/video_runs/mamba_

/tmp/ipykernel_8453/1845424149.py:153: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers)


saved /content/dogsproject/cleanup_detection/models/video_runs/transformer_cls.mp4 | mean_p=0.059 cleanup_rate=0.058 scored=465

=== transformer_wide (transformer) ===
saved /content/dogsproject/cleanup_detection/models/video_runs/transformer_wide.mp4 | mean_p=0.000 cleanup_rate=0.000 scored=465

=== temporal_transformer (temporal_transformer) ===


/tmp/ipykernel_8453/1845424149.py:181: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers)


saved /content/dogsproject/cleanup_detection/models/video_runs/temporal_transformer.mp4 | mean_p=0.184 cleanup_rate=0.185 scored=465

=== VIDEO INFERENCE ALL ARCHITECTURES ===
                 tag                 arch  train_f1  train_accuracy                                                                       output_path   mean_p  cleanup_rate  n_scored_windows  n_cleanup_windows  is_best
           lstm_base                 lstm       1.0             1.0            /content/dogsproject/cleanup_detection/models/video_runs/lstm_base.mp4 0.271471      0.273118               465                127     True
         bilstm_base               bilstm       1.0             1.0          /content/dogsproject/cleanup_detection/models/video_runs/bilstm_base.mp4 0.259893      0.260215               465                121    False
temporal_transformer temporal_transformer       1.0             1.0 /content/dogsproject/cleanup_detection/models/video_runs/temporal_transformer.mp4 0.183856      0.1

## Выгрузка моделей и метрик


In [ ]:
# @title Export zip
try:
    from google.colab import files
except Exception:
    files = None

out_zip = Path('/content') / f"cleanup_models_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
if not Path('/content').exists():
    out_zip = Path(cfg.models_root).parent / out_zip.name

with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in Path(cfg.models_root).rglob('*'):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(Path(cfg.models_root).parent)))
print('zip:', out_zip)
if files is not None:
    files.download(str(out_zip))